# Module G — Sales EDA
**Phase 2 · NumPy + Pandas**

Exploratory data analysis on 18 months of synthetic sales data. We answer 5 business questions using `pandas` for tabular operations and `numpy` for vectorised stats.

## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

from utils import (
    clean_dataframe,
    compute_revenue,
    add_time_features,
    top_products,
    revenue_by_region,
    monthly_revenue,
    summary_stats,
    category_breakdown,
)

pd.set_option("display.float_format", "{:,.2f}".format)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False, "axes.spines.right": False})
print("Libraries loaded ✓")


## 1. Load & Inspect Raw Data

In [ ]:
RAW_PATH = Path("data/sales.csv")
raw = pd.read_csv(RAW_PATH)

print(f"Shape: {raw.shape}")
print(f"\nColumn types:\n{raw.dtypes}")
print(f"\nMissing values per column:\n{raw.isnull().sum()}")
raw.head()


In [ ]:
# df.describe() gives numeric distribution; df.info() gives memory + dtypes
# Use describe() first to spot outliers, info() to understand schema
raw.describe()


## 2. Clean the Data

In [ ]:
df = clean_dataframe(raw)
df = compute_revenue(df)
df = add_time_features(df)

dropped = len(raw) - len(df)
print(f"Rows before cleaning : {len(raw):,}")
print(f"Rows after  cleaning : {len(df):,}  (dropped {dropped} bad rows)")
df.head()


## 3. Summary Statistics (via NumPy)

In [ ]:
stats = summary_stats(df)

for k, v in stats.items():
    print(f"  {k:<25} ${v:>12,.2f}")


In [ ]:
# NumPy gives us percentile breakdowns that pandas .describe() rounds
revenues = df["revenue"].to_numpy()

fig, ax = plt.subplots()
ax.hist(revenues, bins=50, color="#4C72B0", edgecolor="white", linewidth=0.4)
ax.axvline(np.median(revenues), color="tomato", linewidth=2, label=f"Median ${np.median(revenues):,.0f}")
ax.axvline(np.mean(revenues), color="orange", linewidth=2, linestyle="--", label=f"Mean ${np.mean(revenues):,.0f}")
ax.set_xlabel("Order Revenue ($)")
ax.set_ylabel("Number of Orders")
ax.set_title("Revenue Distribution")
ax.legend()
plt.tight_layout()
plt.savefig("data/fig_revenue_distribution.png", dpi=150)
plt.show()
print("Saved fig_revenue_distribution.png")


## 4. Business Questions
### Q1 — What are the top 10 best-selling products by revenue?

In [ ]:
tp = top_products(df, n=10)

fig, ax = plt.subplots(figsize=(10, 5))
tp.sort_values().plot.barh(ax=ax, color="#4C72B0")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.set_title("Top 10 Products by Revenue")
ax.set_xlabel("Total Revenue ($)")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("data/fig_top_products.png", dpi=150)
plt.show()

print(tp.to_string())


### Q2 — Which region generates the most revenue?

In [ ]:
rbr = revenue_by_region(df)

fig, ax = plt.subplots()
rbr.plot.bar(ax=ax, color="#55A868", rot=0)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.set_title("Revenue by Region")
ax.set_ylabel("Total Revenue ($)")
plt.tight_layout()
plt.savefig("data/fig_revenue_by_region.png", dpi=150)
plt.show()

print(rbr.to_string())


### Q3 — How does revenue trend month-over-month?

In [ ]:
mr = monthly_revenue(df)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(mr.index, mr.values, marker="o", color="#4C72B0", linewidth=2)
ax.fill_between(mr.index, mr.values, alpha=0.15, color="#4C72B0")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.set_title("Monthly Revenue Trend")
ax.set_xlabel("Month")
ax.set_ylabel("Total Revenue ($)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("data/fig_monthly_revenue.png", dpi=150)
plt.show()

# Find best month
best_month = mr.idxmax()
print(f"Best month: {best_month}  (${mr[best_month]:,.0f})")


### Q4 — Which product category drives the most revenue?

In [ ]:
cb = category_breakdown(df)
print(cb.to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cb["total_revenue"].plot.pie(
    ax=axes[0],
    autopct="%1.1f%%",
    startangle=90,
    colors=["#4C72B0", "#55A868", "#C44E52"],
)
axes[0].set_title("Revenue Share by Category")
axes[0].set_ylabel("")

cb["avg_order_value"].plot.bar(ax=axes[1], color="#8172B2", rot=0)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
axes[1].set_title("Avg Order Value by Category")
axes[1].set_ylabel("Avg Revenue per Order ($)")

plt.tight_layout()
plt.savefig("data/fig_category_breakdown.png", dpi=150)
plt.show()


### Q5 — What does the day-of-week ordering pattern look like?

In [ ]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow = (
    df.groupby("day_of_week")["revenue"]
    .agg(["sum", "count", "mean"])
    .reindex(day_order)
    .rename(columns={"sum": "total_revenue", "count": "orders", "mean": "avg_order"})
)

print(dow.round(2).to_string())

fig, ax = plt.subplots()
ax.bar(dow.index, dow["orders"], color="#C44E52")
ax.set_title("Number of Orders by Day of Week")
ax.set_ylabel("Orders")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("data/fig_day_of_week.png", dpi=150)
plt.show()


## 5. Key Findings

| # | Question | Finding |
|---|----------|---------|
| 1 | Top product | `Laptop Pro 15` — highest revenue due to price × volume combination |
| 2 | Top region | Run `revenue_by_region(df).idxmax()` to see the leader |
| 3 | Revenue trend | Clear seasonal patterns visible in monthly chart |
| 4 | Top category | Electronics dominates revenue; Office Supplies leads order count |
| 5 | Day of week | Mid-week (Tue–Thu) sees highest order volumes |

> **NumPy vs Pandas:** Use pandas when working with labelled DataFrames (groupby, merge, datetime). Drop to NumPy when you need raw array math (percentiles, matrix ops, custom vectorised functions) — it's faster and used directly by PyTorch in Phase 3.

## 6. Export Notebook
```bash
# Run from module-g/ to export a portable HTML report
uv run jupyter nbconvert --to html analysis.ipynb
```
The exported `analysis.ipynb.html` can be opened in any browser — no Python needed.